# Dans ce notebook on va comparer la trajectoire de Strava et notre Trajectoire

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import sys
import os

# Ajoute le dossier parent au chemin Python
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from utils.correction_erreur_systematique import transformation_csv_dataframe
rayon_terre = 6371000 #rayon de la terre en m

### Lecture et exploration des données 
Dans cette partie on recupère les deux CSV correspondant qu'on explore et plote

In [ ]:
own_data_brut= transformation_csv_dataframe("/home/barry/pi_itineraire/IMU_GPS/data/test_mobile_2_26_250_26.csv")
own_data_kalman = transformation_csv_dataframe("/home/barry/pi_itineraire/IMU_GPS/data/acq_v2_Kalman_26_250_26.csv")
data_strava = transformation_csv_dataframe("/home/barry/pi_itineraire/Strava/2_26_250_26.csv")

In [ ]:
print(own_data_brut.head())
print(data_strava.head())

In [ ]:
# Filtre les lignes GPS invalides AVANT de sélectionner les colonnes
# GPS Valid == 1 + Fix Quality >= 1 : position GPS confirmée
own_data2 = own_data_brut[
    (own_data_brut['GPS Valid'] == 1)
][['Latitude', 'Longitude', 'UTC']].copy()

# Le fichier Kalman n'a pas de colonne GPS Valid → filtre sur les bornes géographiques
# (lat/lon hors [-90,90]/[-180,180] = divergence Kalman certaine)
own_data3 = own_data_kalman[['Latitude', 'Longitude', 'UTC']].copy()

nb_ligne_onw_brut, _   = own_data2.shape
nb_ligne_own_kalman, _ = own_data3.shape
nb_ligne_strava, _     = data_strava.shape
print('Nombre de lignes données brutes  :', nb_ligne_onw_brut)
print('Nombre de lignes données Kalman  :', nb_ligne_own_kalman)
print('Nombre de lignes Strava          :', nb_ligne_strava)


## Synchronisation des deux pour avoir le même time selon Strava
On transforme l'UTC qui est en string en datetime pour pouvoir faire un comparaison pour la synchronisation

In [ ]:
own_data2['UTC'] = pd.to_datetime(own_data2['UTC'], format='%Y/%m/%d - %H:%M:%S', errors='coerce').copy()
own_data2 = own_data2.dropna()
own_data3['UTC'] = pd.to_datetime(own_data3['UTC'],format='%Y/%m/%d - %H:%M:%S', errors='coerce').copy()
own_data3=own_data3.dropna()
data_strava['UTC'] = pd.to_datetime(data_strava['UTC'],format='%Y/%m/%d - %H:%M:%S', errors='coerce').copy()
data_strava=data_strava.dropna()


print(own_data2.describe())
print(own_data3.describe())
print(data_strava.describe())

#### Fonction de synchronisation des données à une fréquence de 1(Hz)
On prend pour le brut et le kalman le prémière acuqistion de chaque séconde 

In [ ]:
def sync(df1, df2, df3, col_time='UTC'):
    dfs = [df1.copy(), df2.copy(), df3.copy()]
    result = []
    for df in dfs:
        df[col_time] = pd.to_datetime(df[col_time])
        df[col_time] = df[col_time].dt.floor('s')
        
        # Garder uniquement la première acquisition de chaque seconde
        df = df.drop_duplicates(subset=[col_time], keep='first')
        df = df.set_index(col_time)
        result.append(df)

    # Intersection des trois index
    common = result[0].index
    for df in result[1:]:
        common = common.intersection(df.index)

    print(f"Points communs (3 df): {len(common)}")
    print(f"Plage temporelle: {common.min()} → {common.max()}")
    return tuple(df.loc[common].reset_index() for df in result)

In [ ]:
own_data_brut_sync,own_data_kalman_sync , data_strava_sync =sync(own_data2,own_data3,data_strava)
print("data brut",own_data_brut_sync.describe())
print("data strava",data_strava_sync.describe())
print("data kalman",own_data_kalman_sync)

#### Merge des trois dataFrame
On recrée un dataFrame pour stocker la latitude et la longitude de chacun des trois df synchronisés

In [ ]:
df_merged = pd.concat([
    own_data_brut_sync,
    own_data_kalman_sync[['Latitude', 'Longitude']].rename(
        columns={'Latitude': 'Lat_kalman', 'Longitude': 'Lon_kalman'}
    ),
    data_strava_sync[['Latitude', 'Longitude']].rename(
        columns={'Latitude': 'Lat_strava', 'Longitude': 'Lon_strava'}
    )
], axis=1).drop(columns=['UTC'])

df_merged.describe()

#### Calcul de la distane entre chaque point pour pouvoir mesuré l'erreur
on base notre mesure surle calcul de la distance entre les deux points 

In [ ]:
rayon_terre = 6371000  # en mètres

def haversine(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    a = np.clip(a, 0, 1)  # sécurité numérique
    return 2 * rayon_terre * np.arcsin(np.sqrt(a))
    
df_merge_rmse = df_merged.copy()
df_merged['dist_brut_kalman'] = haversine(
    df_merged['Latitude'], df_merged['Longitude'],
    df_merged['Lat_kalman'], df_merged['Lon_kalman']
)

df_merged['dist_brut_strava'] = haversine(
    df_merged['Latitude'], df_merged['Longitude'],
    df_merged['Lat_strava'], df_merged['Lon_strava']
)

df_merged['dist_kalman_strava'] = haversine(
    df_merged['Lat_kalman'], df_merged['Lon_kalman'],
    df_merged['Lat_strava'], df_merged['Lon_strava']
)

df_merged.describe()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

distances = {
    'Brut vs Strava':   df_merged['dist_brut_strava'],
    'Kalman vs Strava': df_merged['dist_kalman_strava'],
    'Brut vs Kalman':   df_merged['dist_brut_kalman'],
}

fig, axes = plt.subplots(3, 2, figsize=(14, 14))

for i, (label, serie) in enumerate(distances.items()):
    moyenne    = serie.mean()
    ecart_type = serie.std()
    mediane    = serie.median()

    print(f"=== {label} ===")
    print(f"  Moyenne    : {moyenne:.4f} m")
    print(f"  Écart-type : {ecart_type:.4f} m")
    print(f"  Médiane    : {mediane:.4f} m")
    print(f"  Min        : {serie.min():.4f} m")
    print(f"  Max        : {serie.max():.4f} m\n")

    # Nuage de points
    axes[i, 0].scatter(range(len(serie)), serie, s=1, alpha=0.4, color='steelblue')
    axes[i, 0].axhline(moyenne,               color='red',    linestyle='--', label=f'Moyenne = {moyenne:.3f} m')
    axes[i, 0].axhline(moyenne + ecart_type,  color='orange', linestyle=':',  label=f'±1σ = {ecart_type:.3f} m')
    axes[i, 0].axhline(moyenne - ecart_type,  color='orange', linestyle=':')
    axes[i, 0].set_xlabel("Index temporel")
    axes[i, 0].set_ylabel("Distance (m)")
    axes[i, 0].set_title(f"Écart — {label}")
    axes[i, 0].legend()

    # Histogramme
    axes[i, 1].hist(serie, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
    axes[i, 1].axvline(moyenne,              color='red',    linestyle='--', label=f'Moyenne = {moyenne:.3f} m')
    axes[i, 1].axvline(moyenne + ecart_type, color='orange', linestyle=':',  label='+1σ')
    axes[i, 1].axvline(moyenne - ecart_type, color='orange', linestyle=':',  label='-1σ')
    axes[i, 1].set_xlabel("Distance (m)")
    axes[i, 1].set_ylabel("Fréquence")
    axes[i, 1].set_title(f"Distribution — {label}")
    axes[i, 1].legend()

plt.tight_layout()
plt.show()

### Implémentation RMSE pour une deuxième comparaison 

In [ ]:
rmse_lat_lon = {
    'brut_vs_kalman': {
        'lat': np.sqrt(np.mean((df_merged['Latitude'] - df_merged['Lat_kalman'])**2)),
        'lon': np.sqrt(np.mean((df_merged['Longitude'] - df_merged['Lon_kalman'])**2)),
    },
    'brut_vs_strava': {
        'lat': np.sqrt(np.mean((df_merged['Latitude'] - df_merged['Lat_strava'])**2)),
        'lon': np.sqrt(np.mean((df_merged['Longitude'] - df_merged['Lon_strava'])**2)),
    },
    'kalman_vs_strava': {
        'lat': np.sqrt(np.mean((df_merged['Lat_kalman'] - df_merged['Lat_strava'])**2)),
        'lon': np.sqrt(np.mean((df_merged['Lon_kalman'] - df_merged['Lon_strava'])**2)),
    },
}

for label, vals in rmse_lat_lon.items():
    print(f"RMSE {label:25s} → lat: {vals['lat']:.6f}°  lon: {vals['lon']:.6f}°")